# LeetCode 3349 — Adjacent Increasing Subarrays Detection I bằng Python

Link đề: https://leetcode.com/problems/adjacent-increasing-subarrays-detection-i/ · Độ khó: **Easy** · Chủ đề: Array (mảng), duyệt một lượt, run-length

Skeleton LeetCode:

```python
class Solution:
    def hasIncreasingSubarrays(self, nums: List[int], k: int) -> bool:
        ...
```

Ghi chú: mọi đáp án trong notebook đã được kiểm chứng bằng code (đối chiếu giữa các cách và với một hàm kiểm tra độc lập trên hàng nghìn input ngẫu nhiên có seed). Hai ví dụ chính lấy từ đề; những ví dụ tự dựng thêm có ghi rõ "tự dựng" — nếu muốn khớp từng chữ thì nên đối chiếu lại đề gốc. Khi chạy ngoài LeetCode, ta thêm `from typing import List` (LeetCode đã nhập sẵn).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: tìm hai **mảng con liên tiếp** (subarray) **cùng độ dài `k`**, **nằm sát nhau** (mảng thứ hai bắt đầu ngay sau khi mảng thứ nhất kết thúc: `b = a + k`), và **cả hai đều tăng ngặt**.
2. Nắm các công cụ Python: cắt lát `nums[a:a+k]`, kiểm tra tăng ngặt bằng `all(... for ...)` và `zip`, giới hạn `range(len(nums) - 2*k + 1)`, `enumerate`, `min`/`max`, phép chia nguyên `//`, và tổng tiền tố (prefix sum).
3. Đi từ cách **vét cạn thử từng vị trí `a`** → **mảng độ dài đoạn tăng kết thúc/bắt đầu tại mỗi chỉ số** → **một lượt duyệt giữ `cur`, `pre`, `mx`** → **tách thành các đoạn tăng (run-length)** → **tổng tiền tố trên mảng 0/1**.
4. Hiểu mấu chốt: hai mảng con liền nhau cùng tăng ngặt có hai dạng — **cắt đôi một đoạn tăng dài** (cần độ dài `>= 2k`) hoặc **nằm trên hai đoạn tăng kề nhau** (cần cả hai đoạn `>= k`). Gộp lại: độ dài lớn nhất `mx = max(cur // 2, min(pre, cur))` rồi so `mx >= k`.
5. Tránh các lỗi kinh điển: dùng `<=` thay `<` (tăng ngặt chứ không phải không giảm), sai biên `range` (quên `+ 1`), chỉ xét `cur >= 2k` mà quên trường hợp hai đoạn kề nhau, và quên cập nhật phần tử cuối mảng.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums` độ dài `n` và số nguyên `k`. Hãy xác định xem có tồn tại **hai** mảng con **kề nhau** độ dài `k`, sao cho **cả hai đều tăng ngặt** hay không. Cụ thể, kiểm tra có hai chỉ số `a < b` sao cho:

- `nums[a..a+k-1]` và `nums[b..b+k-1]` đều **tăng ngặt** (mỗi phần tử **lớn hơn hẳn** phần tử đứng trước);
- hai mảng con **kề nhau**: `b = a + k`.

Trả về `true` nếu có, ngược lại `false`.

**Ví dụ 1**

- Input: `nums = [2,5,7,8,9,2,3,4,3,1], k = 3`
- Output: `true`
- Giải thích: mảng con bắt đầu ở chỉ số `2` là `[7, 8, 9]` (tăng ngặt), mảng con bắt đầu ở chỉ số `5` là `[2, 3, 4]` (tăng ngặt), và `5 = 2 + 3` nên chúng kề nhau.

**Ví dụ 2**

- Input: `nums = [1,2,3,4,4,4,4,5,6,7], k = 5`
- Output: `false`

**Ràng buộc**

- `2 <= nums.length <= 100`
- `1 < 2 * k <= nums.length`
- `-1000 <= nums[i] <= 1000`

In [1]:
from typing import List
import random

examples = [
    ([2, 5, 7, 8, 9, 2, 3, 4, 3, 1], 3, True),
    ([1, 2, 3, 4, 4, 4, 4, 5, 6, 7], 5, False),
]

def is_strictly_increasing(arr):
    return all(arr[i] < arr[i + 1] for i in range(len(arr) - 1))

for nums, k, expected in examples:
    print(nums, "k =", k, "kỳ vọng", expected)
    # đối chiếu nhanh: thử mọi a
    found = any(
        is_strictly_increasing(nums[a:a + k]) and is_strictly_increasing(nums[a + k:a + 2 * k])
        for a in range(len(nums) - 2 * k + 1)
    )
    print("  ->", found)
    assert found == expected

[2, 5, 7, 8, 9, 2, 3, 4, 3, 1] k = 3 kỳ vọng True
  -> True
[1, 2, 3, 4, 4, 4, 4, 5, 6, 7] k = 5 kỳ vọng False
  -> False


## 3. Kiến thức nền

### 3.1. Mảng con (subarray) và cắt lát `nums[a:a+k]`

**Mảng con** là một đoạn **liên tiếp** của mảng (khác với **dãy con** có thể bỏ phần tử). Trong Python, `nums[a:b]` lấy các phần tử có chỉ số `a, a+1, ..., b-1` — **không gồm `b`**. Vậy mảng con độ dài `k` bắt đầu ở `a` là `nums[a:a+k]`, và mảng con kề ngay sau nó là `nums[a+k:a+2*k]`.

In [2]:
nums = [2, 5, 7, 8, 9, 2, 3, 4, 3, 1]
a, k = 2, 3
print(nums[a:a + k])            # [7, 8, 9]
print(nums[a + k:a + 2 * k])    # [2, 3, 4]
print(nums[0:0], nums[3:4])     # [] [8]  (lát rỗng / một phần tử)

[7, 8, 9]
[2, 3, 4]
[] [8]


### 3.2. "Tăng ngặt" và cách kiểm tra bằng `all` + `zip`

**Tăng ngặt:** mỗi phần tử **nhỏ hơn hẳn** phần tử kế tiếp (`<`). `[1,2,2]` **không** tăng ngặt vì `2 < 2` sai (nó chỉ là "không giảm"). Mảng có `0` hoặc `1` phần tử thì tăng ngặt **tầm thường** (không có cặp nào vi phạm).

Kiểm tra: mọi cặp kề `(x, y)` đều có `x < y`. `zip(arr, arr[1:])` ghép các cặp kề nhau; `all(...)` trả `True` nếu mọi điều kiện đúng.

In [3]:
def increasing(arr):
    return all(x < y for x, y in zip(arr, arr[1:]))

print(list(zip([7, 8, 9], [7, 8, 9][1:])))   # [(7, 8), (8, 9)]
print(increasing([7, 8, 9]))     # True
print(increasing([1, 2, 2]))     # False (2 < 2 sai)
print(increasing([5, 3]))        # False
print(increasing([4]))           # True  (một phần tử: không có cặp nào sai)
print(increasing([]))            # True

[(7, 8), (8, 9)]
True
False
False
True
True


### 3.3. Chỉ số bắt đầu `a` chạy tới đâu? `range(n - 2*k + 1)`

Cần cả hai mảng con nằm trọn trong mảng: mảng thứ hai kết thúc ở chỉ số `a + 2k - 1 <= n - 1`, tức `a <= n - 2k`. Vậy `a` chạy `0, 1, ..., n - 2k` — trong Python là `range(n - 2*k + 1)` (nhớ `+ 1` vì `range` không gồm điểm cuối).

In [4]:
n, k = 10, 3
print(list(range(n - 2 * k + 1)))    # [0, 1, 2, 3, 4]  -> a lớn nhất là 4 (mảng con thứ hai: chỉ số 7..9)
print(list(range(n - 2 * k)))        # [0, 1, 2, 3]     -> THIẾU a = 4 (lỗi sai biên!)

n, k = 6, 3
print(list(range(n - 2 * k + 1)))    # [0] : chỉ có một cách chọn

[0, 1, 2, 3, 4]
[0, 1, 2, 3]
[0]


### 3.4. Đoạn tăng liên tiếp (run) và độ dài của nó

Ta cắt mảng thành các **đoạn tăng cực đại** (run): đoạn bị ngắt tại chỗ nào `nums[i] >= nums[i+1]`. Ví dụ `[2,5,7,8,9,2,3,4,3,1]` cho các đoạn `[2,5,7,8,9]`, `[2,3,4]`, `[3]`, `[1]` với độ dài `5, 3, 1, 1`. Các đoạn này **phủ hết mảng, không chồng lên nhau**, và đoạn sau bắt đầu ngay chỗ đoạn trước kết thúc.

In [5]:
def split_runs(nums):
    runs = []
    cur = [nums[0]]
    for x in nums[1:]:
        if x > cur[-1]:
            cur.append(x)
        else:
            runs.append(cur)
            cur = [x]
    runs.append(cur)
    return runs

r = split_runs([2, 5, 7, 8, 9, 2, 3, 4, 3, 1])
print(r)                       # [[2, 5, 7, 8, 9], [2, 3, 4], [3], [1]]
print([len(x) for x in r])     # [5, 3, 1, 1]

[[2, 5, 7, 8, 9], [2, 3, 4], [3], [1]]
[5, 3, 1, 1]


### 3.5. Tổng tiền tố (prefix sum) trên mảng 0/1

Đặt `up[i] = 1` nếu `nums[i] < nums[i+1]`, ngược lại `0` (với `i = 0..n-2`). Mảng con `nums[a..a+k-1]` tăng ngặt **khi và chỉ khi** tất cả `k - 1` cặp kề bên trong đều tăng, tức `up[a] + ... + up[a+k-2] == k - 1`. Tổng một đoạn lấy được trong `O(1)` bằng **tổng tiền tố** `pre[i+1] = pre[i] + up[i]`: `sum(up[l..r]) = pre[r+1] - pre[l]`.

In [6]:
nums = [2, 5, 7, 8, 9, 2, 3, 4, 3, 1]
up = [1 if nums[i] < nums[i + 1] else 0 for i in range(len(nums) - 1)]
print(up)                      # [1, 1, 1, 1, 0, 1, 1, 0, 0]

pre = [0]
for v in up:
    pre.append(pre[-1] + v)
print(pre)                     # [0, 1, 2, 3, 4, 4, 5, 6, 6, 6]

def sum_range(l, r):           # tổng up[l..r]
    return pre[r + 1] - pre[l]

print(sum_range(2, 3))         # 2 -> mảng con nums[2..4] = [7,8,9] có k-1 = 2 cặp tăng, tăng ngặt
print(sum_range(5, 6))         # 2 -> nums[5..7] = [2,3,4] tăng ngặt

[1, 1, 1, 1, 0, 1, 1, 0, 0]
[0, 1, 2, 3, 4, 4, 5, 6, 6, 6]
2
2


### 3.6. `//` và `min`/`max`

- `cur // 2` là nửa độ dài (chia nguyên, làm tròn xuống): đoạn tăng dài `cur` cắt đôi được hai mảng con **mỗi mảng `cur // 2` phần tử**. Ví dụ `5 // 2 = 2`.
- `min(pre, cur)`: hai đoạn kề nhau độ dài `pre` và `cur` có thể lấy hai mảng con độ dài `min(pre, cur)` (lấy phần cuối của đoạn trước và phần đầu của đoạn sau).
- `max(...)`: giữ độ dài mảng con lớn nhất tìm được cho tới giờ.

In [7]:
print(5 // 2, 4 // 2, 3 // 2, 1 // 2)    # 2 2 1 0
print(min(5, 3), min(1, 4))              # 3 1
print(max(0, 2, 3))                      # 3

2 2 1 0
3 1
3


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Với `nums = [2,5,7,8,9,2,3,4,3,1]`, `k = 3`, vẽ mũi tên giữa các phần tử kề: `<` là tăng, `>=` là ngắt.

```
2 < 5 < 7 < 8 < 9 >= 2 < 3 < 4 >= 3 >= 1
```

Các đoạn tăng cực đại: `[2,5,7,8,9]` (dài `5`), `[2,3,4]` (dài `3`), `[3]` (dài `1`), `[1]` (dài `1`).

Hai mảng con kề nhau cùng tăng ngặt độ dài `k = 3` có thể:

1. **Cùng nằm trong một đoạn tăng**: đoạn dài `L` chứa được hai mảng con kề nhau độ dài `k` khi `L >= 2k`. Đoạn `[2,5,7,8,9]` dài `5 < 6` nên **không**.
2. **Nằm ở hai đoạn tăng kề nhau**: mảng thứ nhất là `k` phần tử **cuối** của đoạn trước, mảng thứ hai là `k` phần tử **đầu** của đoạn sau. Cần cả hai đoạn có độ dài `>= k`. `[2,5,7,8,9]` (5) và `[2,3,4]` (3) đều `>= 3` — **có!** Mảng con là `[7,8,9]` và `[2,3,4]`.

Ví dụ 2: `[1,2,3,4,4,4,4,5,6,7]`, `k = 5`: các đoạn dài `4, 1, 1, 4`. Không đoạn nào `>= 10` và không cặp đoạn kề nào đều `>= 5`: `min(4,1)=1`, `min(1,1)=1`, `min(1,4)=1`. Kết quả `false`.

In [8]:
nums = [2, 5, 7, 8, 9, 2, 3, 4, 3, 1]
k = 3
lens = [len(r) for r in split_runs(nums)]
print("độ dài các đoạn:", lens)
print("cắt đôi một đoạn (L // 2):", [L // 2 for L in lens])
print("hai đoạn kề (min):", [min(lens[i], lens[i + 1]) for i in range(len(lens) - 1)])

nums2 = [1, 2, 3, 4, 4, 4, 4, 5, 6, 7]
lens2 = [len(r) for r in split_runs(nums2)]
print(lens2, [L // 2 for L in lens2], [min(lens2[i], lens2[i + 1]) for i in range(len(lens2) - 1)])

độ dài các đoạn: [5, 3, 1, 1]
cắt đôi một đoạn (L // 2): [2, 1, 0, 0]
hai đoạn kề (min): [3, 1, 1]
[4, 1, 1, 4] [2, 0, 0, 2] [1, 1, 1]


## 5. Cách 1 — Brute Force / baseline: thử mọi vị trí `a`

**Ý tưởng.** Với mỗi `a` từ `0` đến `n - 2k`: kiểm tra `nums[a..a+k-1]` tăng ngặt **và** `nums[a+k..a+2k-1]` tăng ngặt, bằng hai vòng lặp nhỏ. Nếu có `a` thoả thì trả `True`; hết vòng lặp mà không thấy thì `False`.

In [9]:
def has_increasing_bruteforce(nums: List[int], k: int) -> bool:
    n = len(nums)
    for a in range(n - 2 * k + 1):
        ok1 = True
        for i in range(a, a + k - 1):           # k - 1 cặp kề trong mảng con thứ nhất
            if nums[i] >= nums[i + 1]:
                ok1 = False
                break
        ok2 = True
        for i in range(a + k, a + 2 * k - 1):   # k - 1 cặp kề trong mảng con thứ hai
            if nums[i] >= nums[i + 1]:
                ok2 = False
                break
        if ok1 and ok2:
            return True
    return False

print(has_increasing_bruteforce([2, 5, 7, 8, 9, 2, 3, 4, 3, 1], 3))     # True
print(has_increasing_bruteforce([1, 2, 3, 4, 4, 4, 4, 5, 6, 7], 5))     # False
print(has_increasing_bruteforce([1, 2], 1))                             # True

True
False
True


**Độ phức tạp (Cách 1).** Có `n - 2k + 1` giá trị `a`, mỗi giá trị kiểm tra `O(k)` cặp. Tổng `O((n - 2k) · k)`; vì `2k <= n` nên không vượt `O(n²)` (cụ thể `<= n²/8`). Bộ nhớ phụ `O(1)`.

**Nút thắt.** Hai vị trí `a` và `a + 1` kiểm tra lại **gần như cùng các cặp kề** — công việc lặp. Với `n <= 100` thì chạy tức thì, nhưng ta vẫn muốn một cách `O(n)`: mỗi cặp kề chỉ cần xét **một lần**.

## 6. Ý tưởng thuật toán / bước đột phá

**Mỗi cặp kề `(nums[i], nums[i+1])` chỉ cần được xét một lần** — "tăng" hay "ngắt" — rồi ta suy ra độ dài các đoạn tăng. Với độ dài đoạn tăng cực đại `L`, ta có kết luận gọn:

- Hai mảng con trong **cùng một đoạn** cho độ dài mảng con lớn nhất `L // 2`.
- Hai mảng con ở **hai đoạn kề** `(pre, cur)` cho độ dài lớn nhất `min(pre, cur)`.

Gọi `mx` = max của mọi giá trị trên. Mảng con độ dài `k` thoả đề tồn tại **khi và chỉ khi** `mx >= k` (vì nếu tìm được hai mảng con kề tăng ngặt độ dài `m >= k` thì ta thu nhỏ về độ dài `k` bằng cách lấy `k` phần tử **cuối** của mảng thứ nhất và `k` phần tử **đầu** của mảng thứ hai: hai mảng này vẫn kề nhau — chỗ nối giữ nguyên — và vẫn tăng ngặt).

Chỉ cần một lượt duyệt, giữ `cur` (độ dài đoạn đang xét), `pre` (độ dài đoạn ngay trước), `mx`.

In [10]:
nums = [2, 5, 7, 8, 9, 2, 3, 4, 3, 1]
lens = [len(r) for r in split_runs(nums)]       # [5, 3, 1, 1]
mx = 0
pre = 0
for cur in lens:
    mx = max(mx, cur // 2, min(pre, cur))
    pre = cur
print("mx =", mx)        # 3
for k in range(1, 6):
    print(f"k={k}: ", mx >= k)

mx = 3
k=1:  True
k=2:  True
k=3:  True
k=4:  False
k=5:  False


## 7. Các cách tiếp theo

### Cách 2 — Mảng `inc_len`: độ dài đoạn tăng bắt đầu tại mỗi chỉ số

**Giải thích từng đoạn.**

1. `inc[i]` = độ dài **đoạn tăng dài nhất bắt đầu ở chỉ số `i`** (đi sang phải). Tính từ phải sang trái: `inc[n-1] = 1`; nếu `nums[i] < nums[i+1]` thì `inc[i] = inc[i+1] + 1`, ngược lại `inc[i] = 1`.
2. Mảng con `nums[a..a+k-1]` tăng ngặt **khi và chỉ khi** `inc[a] >= k`.
3. Vậy chỉ cần tồn tại `a` với `inc[a] >= k` và `inc[a + k] >= k`.

In [11]:
def has_increasing_inclen(nums: List[int], k: int) -> bool:
    n = len(nums)
    inc = [1] * n
    for i in range(n - 2, -1, -1):          # từ phải sang trái
        if nums[i] < nums[i + 1]:
            inc[i] = inc[i + 1] + 1
    for a in range(n - 2 * k + 1):
        if inc[a] >= k and inc[a + k] >= k:
            return True
    return False

print([1, 2, 3, 4, 4, 4, 4, 5, 6, 7], "->", has_increasing_inclen([1, 2, 3, 4, 4, 4, 4, 5, 6, 7], 5))
print(has_increasing_inclen([2, 5, 7, 8, 9, 2, 3, 4, 3, 1], 3))   # True

# xem mảng inc cho ví dụ 1
nums = [2, 5, 7, 8, 9, 2, 3, 4, 3, 1]
inc = [1] * len(nums)
for i in range(len(nums) - 2, -1, -1):
    if nums[i] < nums[i + 1]:
        inc[i] = inc[i + 1] + 1
print(inc)    # [5, 4, 3, 2, 1, 3, 2, 1, 1, 1]

[1, 2, 3, 4, 4, 4, 4, 5, 6, 7] -> False
True
[5, 4, 3, 2, 1, 3, 2, 1, 1, 1]


In [12]:
# Code nộp LeetCode cho Cách 2 (đổi tên lớp thành Solution khi nộp)
class SolutionIncLen:
    def hasIncreasingSubarrays(self, nums: List[int], k: int) -> bool:
        n = len(nums)
        inc = [1] * n
        for i in range(n - 2, -1, -1):
            if nums[i] < nums[i + 1]:
                inc[i] = inc[i + 1] + 1
        for a in range(n - 2 * k + 1):
            if inc[a] >= k and inc[a + k] >= k:
                return True
        return False

print(SolutionIncLen().hasIncreasingSubarrays([2, 5, 7, 8, 9, 2, 3, 4, 3, 1], 3))   # True

True


**Độ phức tạp Cách 2:** **Time `O(n)`** (một lượt dựng `inc`, một lượt kiểm tra), **Space `O(n)`** cho mảng `inc`.

### Cách 3 — Một lượt duyệt với `cur`, `pre`, `mx` (lời giải chính thức)

**Giải thích từng đoạn.**

1. `cur` đếm độ dài đoạn tăng đang đi; mỗi phần tử `cur += 1`.
2. **Gặp điểm ngắt** (phần tử cuối mảng, hoặc `x >= nums[i+1]`): đoạn kết thúc. Cập nhật `mx = max(mx, cur // 2, min(pre, cur))` — cắt đôi đoạn này hoặc ghép với đoạn ngay trước.
3. Chuyển giao: `pre = cur`, `cur = 0` để bắt đầu đoạn mới.
4. Cuối cùng trả `mx >= k`.

In [13]:
def has_increasing_onepass(nums: List[int], k: int) -> bool:
    mx = pre = cur = 0
    n = len(nums)
    for i, x in enumerate(nums):
        cur += 1
        if i == n - 1 or x >= nums[i + 1]:       # hết đoạn tăng
            mx = max(mx, cur // 2, min(pre, cur))
            pre, cur = cur, 0
    return mx >= k

print(has_increasing_onepass([2, 5, 7, 8, 9, 2, 3, 4, 3, 1], 3))     # True
print(has_increasing_onepass([1, 2, 3, 4, 4, 4, 4, 5, 6, 7], 5))     # False
print(has_increasing_onepass([1, 2, 3, 4, 5, 6], 3))                 # True (một đoạn dài 6 = 2k)

True
False
True


In [14]:
# Code nộp LeetCode cho Cách 3
class SolutionOnePass:
    def hasIncreasingSubarrays(self, nums: List[int], k: int) -> bool:
        mx = pre = cur = 0
        for i, x in enumerate(nums):
            cur += 1
            if i == len(nums) - 1 or x >= nums[i + 1]:
                mx = max(mx, cur // 2, min(pre, cur))
                pre, cur = cur, 0
        return mx >= k

print(SolutionOnePass().hasIncreasingSubarrays([1, 2, 3, 4, 5, 6], 3))   # True

True


**Độ phức tạp Cách 3:** **Time `O(n)`**, **Space `O(1)`** (chỉ ba biến). Bonus: `mx` chính là **`k` lớn nhất** cho đáp án `true`.

### Cách 4 — Tách thành các đoạn tăng (run-length) rồi xét

**Giải thích.** Tách mảng thành danh sách độ dài các đoạn tăng `runs` (mục 3.4). Sau đó: trả `True` nếu có một đoạn `L >= 2k`, hoặc có hai đoạn kề `runs[j]`, `runs[j+1]` đều `>= k`. Cách này **tách bạch hai bước**, dễ debug hơn cách 3.

In [15]:
def increasing_run_lengths(nums: List[int]) -> List[int]:
    runs = []
    length = 1
    for i in range(1, len(nums)):
        if nums[i] > nums[i - 1]:
            length += 1
        else:
            runs.append(length)
            length = 1
    runs.append(length)
    return runs

def has_increasing_runs(nums: List[int], k: int) -> bool:
    runs = increasing_run_lengths(nums)
    if any(L >= 2 * k for L in runs):
        return True
    return any(runs[j] >= k and runs[j + 1] >= k for j in range(len(runs) - 1))

print(increasing_run_lengths([2, 5, 7, 8, 9, 2, 3, 4, 3, 1]))   # [5, 3, 1, 1]
print(has_increasing_runs([2, 5, 7, 8, 9, 2, 3, 4, 3, 1], 3))   # True
print(has_increasing_runs([1, 2, 3, 4, 4, 4, 4, 5, 6, 7], 5))   # False

[5, 3, 1, 1]
True
False


In [16]:
# Code nộp LeetCode cho Cách 4
class SolutionRuns:
    def hasIncreasingSubarrays(self, nums: List[int], k: int) -> bool:
        runs = []
        length = 1
        for i in range(1, len(nums)):
            if nums[i] > nums[i - 1]:
                length += 1
            else:
                runs.append(length)
                length = 1
        runs.append(length)
        if any(L >= 2 * k for L in runs):
            return True
        return any(runs[j] >= k and runs[j + 1] >= k for j in range(len(runs) - 1))

print(SolutionRuns().hasIncreasingSubarrays([2, 5, 7, 8, 9, 2, 3, 4, 3, 1], 3))   # True

True


**Độ phức tạp Cách 4:** **Time `O(n)`**, **Space `O(n)`** (danh sách `runs`, tối đa `n` đoạn).

### Cách 5 — Mảng 0/1 + tổng tiền tố

**Giải thích.** Theo mục 3.5: dựng `up[i]` và tổng tiền tố `pre`. Mảng con bắt đầu ở `a` tăng ngặt khi `pre[a + k - 1] - pre[a] == k - 1` (tổng `up[a..a+k-2]`). Cần điều kiện này cho cả `a` và `a + k`.

In [17]:
def has_increasing_prefix(nums: List[int], k: int) -> bool:
    n = len(nums)
    pre = [0] * n                       # pre[i] = tổng up[0..i-1]
    for i in range(n - 1):
        pre[i + 1] = pre[i] + (1 if nums[i] < nums[i + 1] else 0)

    def good(a):                        # nums[a..a+k-1] tăng ngặt?
        return pre[a + k - 1] - pre[a] == k - 1

    return any(good(a) and good(a + k) for a in range(n - 2 * k + 1))

print(has_increasing_prefix([2, 5, 7, 8, 9, 2, 3, 4, 3, 1], 3))     # True
print(has_increasing_prefix([1, 2, 3, 4, 4, 4, 4, 5, 6, 7], 5))     # False
print(has_increasing_prefix([5, 5], 1))                             # True (k = 1)

True
False
True


In [18]:
# Code nộp LeetCode cho Cách 5
class SolutionPrefix:
    def hasIncreasingSubarrays(self, nums: List[int], k: int) -> bool:
        n = len(nums)
        pre = [0] * n
        for i in range(n - 1):
            pre[i + 1] = pre[i] + (1 if nums[i] < nums[i + 1] else 0)
        for a in range(n - 2 * k + 1):
            if pre[a + k - 1] - pre[a] == k - 1 and pre[a + 2 * k - 1] - pre[a + k] == k - 1:
                return True
        return False

print(SolutionPrefix().hasIncreasingSubarrays([5, 5], 1))   # True

True


**Độ phức tạp Cách 5:** **Time `O(n)`**, **Space `O(n)`**. Hữu ích khi phải trả lời **nhiều truy vấn** "đoạn `[l, r]` có tăng ngặt không?" trên cùng một mảng.

## 8. TRACE từng bước

Trace Cách 3 trên `nums = [2,5,7,8,9,2,3,4,3,1]`, `k = 3`. Mỗi bước in `i`, `x`, `cur` sau khi cộng, có ngắt đoạn không, và `pre`, `mx` sau cập nhật.

In [19]:
def trace_onepass(nums: List[int], k: int) -> bool:
    n = len(nums)
    mx = pre = cur = 0
    print(f"nums = {nums}, k = {k}")
    for i, x in enumerate(nums):
        cur += 1
        nxt = nums[i + 1] if i + 1 < n else None
        cut = (i == n - 1) or (x >= nums[i + 1])
        print(f"i={i}, x={x}, next={nxt}, cur={cur}, ngắt đoạn? {cut}", end="")
        if cut:
            cand = (cur // 2, min(pre, cur))
            mx = max(mx, *cand)
            print(f" -> ứng viên (cur//2={cand[0]}, min(pre,cur)={cand[1]}), mx={mx}, pre={cur}, cur=0")
            pre, cur = cur, 0
        else:
            print()
    print("mx =", mx, "-> mx >= k ?", mx >= k)
    return mx >= k

trace_onepass([2, 5, 7, 8, 9, 2, 3, 4, 3, 1], 3)
print()
trace_onepass([1, 2, 3, 4, 4, 4, 4, 5, 6, 7], 5)

nums = [2, 5, 7, 8, 9, 2, 3, 4, 3, 1], k = 3
i=0, x=2, next=5, cur=1, ngắt đoạn? False
i=1, x=5, next=7, cur=2, ngắt đoạn? False
i=2, x=7, next=8, cur=3, ngắt đoạn? False
i=3, x=8, next=9, cur=4, ngắt đoạn? False
i=4, x=9, next=2, cur=5, ngắt đoạn? True -> ứng viên (cur//2=2, min(pre,cur)=0), mx=2, pre=5, cur=0
i=5, x=2, next=3, cur=1, ngắt đoạn? False
i=6, x=3, next=4, cur=2, ngắt đoạn? False
i=7, x=4, next=3, cur=3, ngắt đoạn? True -> ứng viên (cur//2=1, min(pre,cur)=3), mx=3, pre=3, cur=0
i=8, x=3, next=1, cur=1, ngắt đoạn? True -> ứng viên (cur//2=0, min(pre,cur)=1), mx=3, pre=1, cur=0
i=9, x=1, next=None, cur=1, ngắt đoạn? True -> ứng viên (cur//2=0, min(pre,cur)=1), mx=3, pre=1, cur=0
mx = 3 -> mx >= k ? True

nums = [1, 2, 3, 4, 4, 4, 4, 5, 6, 7], k = 5
i=0, x=1, next=2, cur=1, ngắt đoạn? False
i=1, x=2, next=3, cur=2, ngắt đoạn? False
i=2, x=3, next=4, cur=3, ngắt đoạn? False
i=3, x=4, next=4, cur=4, ngắt đoạn? True -> ứng viên (cur//2=2, min(pre,cur)=0), mx=2, pre=4, cur=0
i=4

False

Bảng trace cho `nums = [2,5,7,8,9,2,3,4,3,1]`, `k = 3` (chỉ các bước ngắt đoạn):

| i | x | `cur` khi ngắt | `cur // 2` | `min(pre, cur)` | `mx` sau | `pre` mới |
|---|---|---|---|---|---|---|
| 4 | 9 | 5 | 2 | 0 | 2 | 5 |
| 7 | 4 | 3 | 1 | 3 | 3 | 3 |
| 8 | 3 | 1 | 0 | 1 | 3 | 1 |
| 9 | 1 | 1 | 0 | 1 | 3 | 1 |

`mx = 3 >= k = 3` nên trả `True`. Đáng chú ý bước `i = 7`: đoạn `[2,3,4]` (dài 3) ghép với đoạn trước `[2,5,7,8,9]` (dài 5) cho `min = 3`. Còn ví dụ 2: `mx` chỉ lên tới `2 < 5` nên `False`.

## 9. Code nộp LeetCode

Bản nộp dùng Cách 3 (một lượt, `O(1)` bộ nhớ).

In [20]:
class Solution:
    def hasIncreasingSubarrays(self, nums: List[int], k: int) -> bool:
        mx = pre = cur = 0
        for i, x in enumerate(nums):
            cur += 1
            if i == len(nums) - 1 or x >= nums[i + 1]:
                mx = max(mx, cur // 2, min(pre, cur))
                pre, cur = cur, 0
        return mx >= k

sol = Solution()
print(sol.hasIncreasingSubarrays([2, 5, 7, 8, 9, 2, 3, 4, 3, 1], 3))     # True
print(sol.hasIncreasingSubarrays([1, 2, 3, 4, 4, 4, 4, 5, 6, 7], 5))     # False

True
False


## 10. Vì sao thuật toán đúng

- **Phủ kín, không chồng:** các đoạn tăng cực đại chia mảng thành các khúc liên tiếp; đoạn sau bắt đầu ngay vị trí đoạn trước kết thúc (`nums[i] >= nums[i+1]` là chỗ cắt).
- **Hai dạng duy nhất:** một mảng con tăng ngặt **không thể vắt qua chỗ ngắt**, nên nó nằm trọn trong một đoạn. Hai mảng con **kề nhau** hoặc nằm cùng một đoạn (khi đó đoạn có độ dài `>= 2k`, tức `cur // 2 >= k`), hoặc nằm ở **hai đoạn khác nhau**; vì chúng kề nhau nên hai đoạn đó phải **liền kề**, mảng thứ nhất là phần cuối đoạn trước, mảng thứ hai là phần đầu đoạn sau, cần `min(pre, cur) >= k`.
- **Chiều ngược lại:** nếu `cur // 2 >= k` (hay `min(pre, cur) >= k`) thì ta **dựng được** hai mảng con kề nhau độ dài `k` tăng ngặt: lấy `k` phần tử đầu và `k` phần tử kế (hay `k` cuối của đoạn trước + `k` đầu của đoạn sau).
- **Gom `max`:** `mx` là độ dài lớn nhất khả thi; vì khả thi với `m` thì khả thi với mọi `k <= m` (cắt bớt), nên đáp án là `mx >= k`.
- **Phần tử cuối:** điều kiện `i == n - 1` đảm bảo **đoạn cuối** cũng được tính (nếu không, đoạn cuối sẽ bị bỏ sót).

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)`.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1 — thử mọi `a` | `O((n − 2k) · k)` ≤ `O(n²)` | `O(1)` | đơn giản, hơi lặp |
| 2 — `inc[i]` | `O(n)` | `O(n)` | điều kiện `inc[a] >= k and inc[a+k] >= k` |
| 3 — `cur`, `pre`, `mx` | `O(n)` | `O(1)` | chính thức, tối ưu |
| 4 — run-length | `O(n)` | `O(n)` | hai bước tách bạch |
| 5 — prefix sum 0/1 | `O(n)` | `O(n)` | tốt cho nhiều truy vấn |

Với `n <= 100` mọi cách đều tức thì; Big-O quan trọng khi mở rộng `n` lớn hơn.

## 12. Bộ test

Ta gom **mọi cách** vào `impls` và còn dùng thêm một **hàm kiểm tra độc lập** (`oracle`) viết bằng cách hoàn toàn khác: `sub == sorted(set(sub))` (mảng tăng ngặt khi và chỉ khi nó bằng chính nó sau khi bỏ trùng và sắp xếp).

In [21]:
def oracle(nums: List[int], k: int) -> bool:
    for a in range(len(nums) - 2 * k + 1):
        s1 = nums[a:a + k]
        s2 = nums[a + k:a + 2 * k]
        if s1 == sorted(set(s1)) and s2 == sorted(set(s2)):
            return True
    return False

impls = {
    "bruteforce": has_increasing_bruteforce,
    "inclen": has_increasing_inclen,
    "onepass": has_increasing_onepass,
    "runs": has_increasing_runs,
    "prefix": has_increasing_prefix,
    "Solution": Solution().hasIncreasingSubarrays,
    "SolutionIncLen": SolutionIncLen().hasIncreasingSubarrays,
    "SolutionOnePass": SolutionOnePass().hasIncreasingSubarrays,
    "SolutionRuns": SolutionRuns().hasIncreasingSubarrays,
    "SolutionPrefix": SolutionPrefix().hasIncreasingSubarrays,
    "oracle": oracle,
}

def check_all(nums, k, expected, note=""):
    for name, f in impls.items():
        got = f(list(nums), k)
        assert got == expected, (name, nums, k, got, expected)
    print(f"OK  k={k}, nums={nums} -> {expected}  {note}")

check_all([2, 5, 7, 8, 9, 2, 3, 4, 3, 1], 3, True, "ví dụ 1")
check_all([1, 2, 3, 4, 4, 4, 4, 5, 6, 7], 5, False, "ví dụ 2")

OK  k=3, nums=[2, 5, 7, 8, 9, 2, 3, 4, 3, 1] -> True  ví dụ 1
OK  k=5, nums=[1, 2, 3, 4, 4, 4, 4, 5, 6, 7] -> False  ví dụ 2


### 12.1. Các trường hợp biên

In [22]:
check_all([1, 1], 1, True, "k = 1: hai mảng con một phần tử luôn tăng ngặt (tầm thường)")
check_all([5, 4, 3, 2], 2, False, "giảm dần: không có mảng con dài 2 tăng ngặt")
check_all([1, 2, 3, 4], 2, True, "một đoạn dài đúng 2k = 4: cắt đôi")
check_all([1, 2, 3], 2, False, "2k = 4 > n nằm ngoài ràng buộc nhưng hàm vẫn phải trả False (tự dựng)")
check_all([1, 2, 3, 4, 5, 6], 3, True, "đoạn dài đúng 2k = 6")
check_all([1, 2, 3, 3, 4, 5], 3, True, "[1,2,3] và [3,4,5]: chỗ nối 3 → 3 không cần tăng, vì hai mảng con là riêng biệt")
check_all([1, 2, 3, 2, 3, 4], 3, True, "hai đoạn kề nhau: [1,2,3] và [2,3,4]")
check_all([1, 2, 2, 3, 4, 5], 3, False, "[1,2],[2,3,4,5]: min(2,4)=2 < 3 và 4//2=2 < 3")
check_all([1000, -1000, 5, 6], 2, False, "biên giá trị: [1000,-1000] không tăng (tự dựng)")
check_all([-1000, -999, 999, 1000], 2, True, "toàn bộ tăng, biên giá trị -1000..1000")
check_all([1, 2, 1, 2, 1, 2, 1, 2], 2, True, "[1,2] rồi [1,2] liên tiếp nhiều lần")
check_all([3, 2, 1, 2, 3, 2], 2, False, "mọi cặp mảng con kề nhau đều có một mảng không tăng (tự dựng)")

OK  k=1, nums=[1, 1] -> True  k = 1: hai mảng con một phần tử luôn tăng ngặt (tầm thường)
OK  k=2, nums=[5, 4, 3, 2] -> False  giảm dần: không có mảng con dài 2 tăng ngặt
OK  k=2, nums=[1, 2, 3, 4] -> True  một đoạn dài đúng 2k = 4: cắt đôi
OK  k=2, nums=[1, 2, 3] -> False  2k = 4 > n nằm ngoài ràng buộc nhưng hàm vẫn phải trả False (tự dựng)
OK  k=3, nums=[1, 2, 3, 4, 5, 6] -> True  đoạn dài đúng 2k = 6
OK  k=3, nums=[1, 2, 3, 3, 4, 5] -> True  [1,2,3] và [3,4,5]: chỗ nối 3 → 3 không cần tăng, vì hai mảng con là riêng biệt
OK  k=3, nums=[1, 2, 3, 2, 3, 4] -> True  hai đoạn kề nhau: [1,2,3] và [2,3,4]
OK  k=3, nums=[1, 2, 2, 3, 4, 5] -> False  [1,2],[2,3,4,5]: min(2,4)=2 < 3 và 4//2=2 < 3
OK  k=2, nums=[1000, -1000, 5, 6] -> False  biên giá trị: [1000,-1000] không tăng (tự dựng)
OK  k=2, nums=[-1000, -999, 999, 1000] -> True  toàn bộ tăng, biên giá trị -1000..1000
OK  k=2, nums=[1, 2, 1, 2, 1, 2, 1, 2] -> True  [1,2] rồi [1,2] liên tiếp nhiều lần
OK  k=2, nums=[3, 2, 1, 2, 3, 2] -> Fal

### 12.2. Phản ví dụ cho các cách làm sai

Ba cách "gần đúng" hay gặp: (a) dùng `<=` thay `<` (xem "không giảm" là tăng); (b) chỉ xét `cur >= 2k` (bỏ qua hai đoạn kề nhau); (c) quên cập nhật đoạn cuối. Ta tìm input làm chúng lệch so với `oracle`.

In [23]:
def wrong_nonstrict(nums, k):               # (a) cho phép bằng nhau
    n = len(nums)
    for a in range(n - 2 * k + 1):
        ok = all(nums[i] <= nums[i + 1] for i in range(a, a + k - 1)) and \
             all(nums[i] <= nums[i + 1] for i in range(a + k, a + 2 * k - 1))
        if ok:
            return True
    return False

def wrong_one_run_only(nums, k):            # (b) chỉ xét cắt đôi một đoạn
    cur = 1
    best = 1
    for i in range(1, len(nums)):
        cur = cur + 1 if nums[i] > nums[i - 1] else 1
        best = max(best, cur)
    return best >= 2 * k

def wrong_forget_last(nums, k):             # (c) quên đoạn cuối
    mx = pre = cur = 0
    for i in range(len(nums) - 1):
        cur += 1
        if nums[i] >= nums[i + 1]:
            mx = max(mx, cur // 2, min(pre, cur))
            pre, cur = cur, 0
    return mx >= k

def find_counterexample(wrong, tries=20000, seed=1):
    rng = random.Random(seed)
    for _ in range(tries):
        n = rng.randint(2, 10)
        k = rng.randint(1, n // 2)
        nums = [rng.randint(0, 4) for _ in range(n)]
        if wrong(nums, k) != oracle(nums, k):
            return nums, k, wrong(nums, k), oracle(nums, k)

for w in (wrong_nonstrict, wrong_one_run_only, wrong_forget_last):
    ce = find_counterexample(w)
    print(w.__name__, "-> (nums, k, sai, đúng) =", ce)
    assert ce is not None

wrong_nonstrict -> (nums, k, sai, đúng) = ([1, 4, 1, 0, 1, 4, 4, 1], 2, True, False)
wrong_one_run_only -> (nums, k, sai, đúng) = ([3, 1], 1, False, True)
wrong_forget_last -> (nums, k, sai, đúng) = ([3, 1], 1, False, True)


### 12.3. Đối chiếu ngẫu nhiên (có seed) và vét cạn các mảng nhỏ

Đầu vào hợp lệ: `2 <= n <= 100`, `1 < 2k <= n`, giá trị trong `[-1000, 1000]`. Ta thử hai loại: (1) mảng ngẫu nhiên giá trị nhỏ để có nhiều đoạn bằng nhau, (2) mảng được dựng từ các đoạn tăng ngẫu nhiên (khiến đáp án `True` xuất hiện thường xuyên). Ngoài ra, vét cạn **mọi** mảng giá trị `0..3` độ dài `2..7`.

In [24]:
import itertools

rng = random.Random(3349)
true_cnt = false_cnt = 0
for trial in range(4000):
    n = rng.randint(2, 100)
    k = rng.randint(1, n // 2)
    mode = rng.choice(["small", "wide", "runs"])
    if mode == "small":
        nums = [rng.randint(-2, 2) for _ in range(n)]
    elif mode == "wide":
        nums = [rng.randint(-1000, 1000) for _ in range(n)]
    else:
        nums = []
        while len(nums) < n:
            L = rng.randint(1, max(1, k + 1))
            start = rng.randint(-1000, 900)
            nums.extend(start + j for j in range(L))
        nums = nums[:n]
    expected = oracle(nums, k)
    for name, f in impls.items():
        assert f(list(nums), k) == expected, (name, nums, k)
    if expected: true_cnt += 1
    else: false_cnt += 1
print("ngẫu nhiên OK; True:", true_cnt, "False:", false_cnt)

total = 0
for n in range(2, 8):
    for tup in itertools.product(range(4), repeat=n):
        for k in range(1, n // 2 + 1):
            expected = oracle(list(tup), k)
            for name, f in impls.items():
                assert f(list(tup), k) == expected, (name, tup, k)
            total += 1
print("vét cạn", total, "trường hợp nhỏ: OK")

ngẫu nhiên OK; True: 1314 False: 2686


vét cạn 64080 trường hợp nhỏ: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Dùng `<=` thay `<` (tăng không ngặt)

Đề nói **tăng ngặt** (strictly increasing): hai phần tử bằng nhau **không** được.

In [25]:
def sai_khong_ngat(nums, k):
    n = len(nums)
    for a in range(n - 2 * k + 1):
        t1 = all(nums[i] <= nums[i + 1] for i in range(a, a + k - 1))      # SAI: <=
        t2 = all(nums[i] <= nums[i + 1] for i in range(a + k, a + 2 * k - 1))
        if t1 and t2:
            return True
    return False

nums, k = [1, 1, 2, 2], 2
print(sai_khong_ngat(nums, k), "(sai)  vs  đúng:", Solution().hasIncreasingSubarrays(nums, k))
assert sai_khong_ngat(nums, k) is True and Solution().hasIncreasingSubarrays(nums, k) is False

True (sai)  vs  đúng: False


**Vì sao sai:** `[1,1]` có `1 <= 1` nhưng `1 < 1` sai nên **không** tăng ngặt. **Sửa:** dùng `<` khi kiểm tra tăng, và `>=` khi phát hiện điểm ngắt.

### Lỗi 2: Sai biên `range` — quên `+ 1`

In [26]:
def sai_bien_range(nums, k):
    n = len(nums)
    for a in range(n - 2 * k):                   # SAI: thiếu + 1, bỏ sót a = n - 2k
        if increasing(nums[a:a + k]) and increasing(nums[a + k:a + 2 * k]):
            return True
    return False

nums, k = [1, 2, 3, 1, 2, 3], 3      # chỉ có đúng một cách chọn: a = 0 = n - 2k
print(sai_bien_range(nums, k), "(sai)  vs  đúng:", Solution().hasIncreasingSubarrays(nums, k))
assert sai_bien_range(nums, k) is False and Solution().hasIncreasingSubarrays(nums, k) is True

False (sai)  vs  đúng: True


**Vì sao sai:** `range(m)` không gồm `m`. Khi `n = 2k` thì `range(0)` rỗng và vòng lặp **không chạy lần nào**. **Sửa:** `range(n - 2 * k + 1)`.

### Lỗi 3: Chỉ xét một đoạn tăng dài `>= 2k`

In [27]:
def sai_chi_mot_doan(nums, k):
    cur = 1
    best = 1
    for i in range(1, len(nums)):
        cur = cur + 1 if nums[i] > nums[i - 1] else 1
        best = max(best, cur)
    return best >= 2 * k        # SAI: bỏ qua trường hợp hai đoạn kề nhau

nums, k = [2, 5, 7, 8, 9, 2, 3, 4, 3, 1], 3
print(sai_chi_mot_doan(nums, k), "(sai)  vs  đúng:", Solution().hasIncreasingSubarrays(nums, k))
assert sai_chi_mot_doan(nums, k) is False and Solution().hasIncreasingSubarrays(nums, k) is True

False (sai)  vs  đúng: True


**Vì sao sai:** hai mảng con kề nhau không nhất thiết cùng nằm trong một đoạn tăng; chúng có thể thuộc **hai đoạn tăng liền kề** (ví dụ 1: `[7,8,9]` | `[2,3,4]`, ở giữa là chỗ ngắt `9 >= 2`). **Sửa:** thêm `min(pre, cur)`.

### Lỗi 4: Quên xử lý đoạn cuối mảng

In [28]:
def sai_quen_doan_cuoi(nums, k):
    mx = pre = cur = 0
    for i in range(len(nums) - 1):          # SAI: bỏ phần tử cuối
        cur += 1
        if nums[i] >= nums[i + 1]:
            mx = max(mx, cur // 2, min(pre, cur))
            pre, cur = cur, 0
    return mx >= k

nums, k = [5, 1, 2, 3, 4], 2          # đoạn cuối [1,2,3,4] dài 4: cắt đôi được [1,2] và [3,4]
print(sai_quen_doan_cuoi(nums, k), "(sai)  vs  đúng:", Solution().hasIncreasingSubarrays(nums, k))
assert sai_quen_doan_cuoi(nums, k) is False and Solution().hasIncreasingSubarrays(nums, k) is True

False (sai)  vs  đúng: True


**Vì sao sai:** đoạn tăng cuối cùng **không có phần tử sau** để "ngắt", nên nếu chỉ cập nhật khi `nums[i] >= nums[i+1]` thì đoạn này bị bỏ. **Sửa:** điều kiện ngắt phải có `i == n - 1 or ...` (hoặc thêm một phần tử "lính canh").

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Mảng con liên tiếp thoả tính chất đơn điệu" → chia thành các đoạn (run)**. Biết độ dài các run là giải quyết được nhiều bài: "Longest Continuous Increasing Subsequence", "Count Subarrays With ...", "Number of Increasing Subarrays".
- **Một lượt duyệt với `cur`/`pre`:** khi chỉ cần quan hệ với đoạn **ngay trước**, đừng lưu mọi đoạn — giữ hai biến là đủ.
- **Điều kiện "cùng một đoạn" vs "hai đoạn kề":** trong bài này luôn tách hai trường hợp `cur // 2` và `min(pre, cur)`.
- **Phần tử cuối mảng:** mỗi khi dùng `nums[i+1]` hoặc "đóng đoạn tại điểm ngắt", hãy tự hỏi đoạn cuối được đóng ở đâu.
- **Prefix sum trên mảng cờ 0/1** biến "mọi cặp trong đoạn đều thoả" thành phép trừ `O(1)`.
- Ràng buộc `n <= 100` cho phép vét cạn, nhưng viết `O(n)` luôn là lựa chọn an toàn.

## 15. Biến thể của bài

1. **Adjacent Increasing Subarrays Detection II** (LeetCode 3350): tìm **`k` lớn nhất** để có hai mảng con kề nhau tăng ngặt. Chính là giá trị `mx` của Cách 3!
2. **Không ngặt** (non-decreasing): đổi `<` thành `<=` ở điều kiện tăng, `>=` thành `>` ở điều kiện ngắt.
3. **Giảm ngặt** hoặc **đơn điệu** (tăng hoặc giảm).
4. **Ba mảng con kề nhau** tăng ngặt: tổng quát hoá: `m` mảng con kề nhau cùng tăng ngặt.

In [29]:
def max_k_adjacent_increasing(nums: List[int]) -> int:
    """Biến thể II: k lớn nhất sao cho có hai mảng con kề nhau độ dài k cùng tăng ngặt."""
    mx = pre = cur = 0
    for i, x in enumerate(nums):
        cur += 1
        if i == len(nums) - 1 or x >= nums[i + 1]:
            mx = max(mx, cur // 2, min(pre, cur))
            pre, cur = cur, 0
    return mx

print(max_k_adjacent_increasing([2, 5, 7, 8, 9, 2, 3, 4, 3, 1]))   # 3
print(max_k_adjacent_increasing([1, 2, 3, 4, 4, 4, 4, 5, 6, 7]))   # 2
print(max_k_adjacent_increasing([5, 4, 3]))                        # 1 (mảng con 1 phần tử luôn tăng ngặt)

# kiểm tra: k lớn nhất khớp với hàm gốc
nums = [2, 5, 7, 8, 9, 2, 3, 4, 3, 1]
m = max_k_adjacent_increasing(nums)
assert Solution().hasIncreasingSubarrays(nums, m) and not Solution().hasIncreasingSubarrays(nums, m + 1)

def has_m_adjacent_increasing(nums, k, m):
    """Biến thể tổng quát: có m mảng con kề nhau, mỗi mảng dài k, đều tăng ngặt?"""
    n = len(nums)
    for a in range(n - m * k + 1):
        if all(increasing(nums[a + j * k:a + (j + 1) * k]) for j in range(m)):
            return True
    return False

print(has_m_adjacent_increasing([1, 2, 3, 4, 5, 6, 7, 8, 9], 3, 3))   # True
print(has_m_adjacent_increasing([1, 2, 3, 4, 5, 6, 7, 8, 9], 3, 4))   # False (cần n >= 12)

3
2
1
True
False


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1 | Thử mọi `a`, kiểm tra hai mảng con | `O((n−2k)·k)` | `O(1)` | `n` nhỏ, muốn code ngay |
| 2 | `inc[i]` = độ dài đoạn tăng bắt đầu tại `i` | `O(n)` | `O(n)` | cần biết độ dài đoạn tại mọi `i` |
| 3 | Một lượt: `cur`, `pre`, `mx` | `O(n)` | `O(1)` | lời giải chuẩn; còn cho ra `k` lớn nhất |
| 4 | Tách run-length rồi xét `L >= 2k` / hai run kề `>= k` | `O(n)` | `O(n)` | dễ debug, hai bước rõ ràng |
| 5 | Prefix sum trên cờ tăng 0/1 | `O(n)` | `O(n)` | nhiều truy vấn đoạn |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `nums[1:4]` với `nums = [4, 5, 6, 7, 8]`; `increasing([3, 3, 4])`; `list(range(7 - 2 * 2 + 1))`; `5 // 2`; `Solution().hasIncreasingSubarrays([1, 2, 3, 4], 2)`.

**Bài 2 (tính tay).** Với `nums = [1, 3, 5, 2, 4, 6, 8, 7]`: liệt kê các đoạn tăng cực đại và độ dài; tính `mx`; trả lời cho `k = 3` và `k = 4`.

**Bài 3 (điền TODO).** Hoàn thành `has_increasing_todo(nums, k)` theo Cách 3.

In [30]:
def has_increasing_todo(nums, k):
    # TODO 1: khởi tạo mx, pre, cur = 0, 0, 0
    # TODO 2: duyệt i, x bằng enumerate(nums); cur += 1
    # TODO 3: nếu i là chỉ số cuối HOẶC x >= nums[i + 1]: đóng đoạn
    #         - cập nhật mx = max(mx, cur // 2, min(pre, cur))
    #         - pre, cur = cur, 0
    # TODO 4: trả về mx >= k
    pass

# Tự làm xong rồi mới mở test:
# assert has_increasing_todo([2, 5, 7, 8, 9, 2, 3, 4, 3, 1], 3) is True
# assert has_increasing_todo([1, 2, 3, 4, 4, 4, 4, 5, 6, 7], 5) is False

**Bài 4 (tự viết lại).** Viết `has_increasing_v2(nums, k)` theo Cách 2 **mà dựng `inc` từ trái sang phải** (tức `inc_end[i]` = độ dài đoạn tăng **kết thúc** ở `i`). Gợi ý: khi đó điều kiện là `inc_end[b - 1] >= k` và `inc_end[b + k - 1] >= k` với `b = a + k`.

**Bài 5 (biến thể / mini challenge).** Viết `count_valid_a(nums, k)` đếm **có bao nhiêu** chỉ số `a` thoả hai mảng con `nums[a..a+k-1]`, `nums[a+k..a+2k-1]` cùng tăng ngặt. Kiểm tra: `count_valid_a([1, 2, 3, 4, 5, 6, 7], 3) == 2` (a = 0 và a = 1). Thử viết bản `O(n)`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `[5, 6, 7]`, `False` (`3 < 3` sai), `[0, 1, 2, 3]`, `2`, `True` (`[1,2]` và `[3,4]`). Bài 3: nhớ điều kiện dừng đoạn có cả `i == len(nums) - 1`.
- **Mức 2 (vừa):** Bài 2: các đoạn `[1,3,5]`, `[2,4,6,8]`, `[7]` dài `3, 4, 1`; `cur // 2`: `1, 2, 0`; `min` kề: `min(3,4) = 3`, `min(4,1) = 1`; `mx = 3`. Vậy `k = 3` là `True`, `k = 4` là `False`. Bài 4: `inc_end[0] = 1; inc_end[i] = inc_end[i-1] + 1 nếu nums[i-1] < nums[i]`.
- **Mức 3 (sâu):** Bài 5: dùng `inc[a] >= k and inc[a + k] >= k` (Cách 2) trong một vòng `for a in range(n - 2*k + 1)` và đếm — vẫn `O(n)`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [31]:
# Bài 1
nums_b1 = [4, 5, 6, 7, 8]
assert nums_b1[1:4] == [5, 6, 7]
assert increasing([3, 3, 4]) is False
assert list(range(7 - 2 * 2 + 1)) == [0, 1, 2, 3]
assert 5 // 2 == 2
assert Solution().hasIncreasingSubarrays([1, 2, 3, 4], 2) is True

# Bài 2
nums_b2 = [1, 3, 5, 2, 4, 6, 8, 7]
assert [len(r) for r in split_runs(nums_b2)] == [3, 4, 1]
assert max_k_adjacent_increasing(nums_b2) == 3
assert Solution().hasIncreasingSubarrays(nums_b2, 3) is True
assert Solution().hasIncreasingSubarrays(nums_b2, 4) is False

# Bài 3
def has_increasing_todo_solution(nums, k):
    mx = pre = cur = 0
    for i, x in enumerate(nums):
        cur += 1
        if i == len(nums) - 1 or x >= nums[i + 1]:
            mx = max(mx, cur // 2, min(pre, cur))
            pre, cur = cur, 0
    return mx >= k

assert has_increasing_todo_solution([2, 5, 7, 8, 9, 2, 3, 4, 3, 1], 3) is True
assert has_increasing_todo_solution([1, 2, 3, 4, 4, 4, 4, 5, 6, 7], 5) is False

# Bài 4: inc_end[i] = độ dài đoạn tăng kết thúc ở i
def has_increasing_v2(nums, k):
    n = len(nums)
    inc_end = [1] * n
    for i in range(1, n):
        if nums[i - 1] < nums[i]:
            inc_end[i] = inc_end[i - 1] + 1
    for a in range(n - 2 * k + 1):
        b = a + k
        if inc_end[b - 1] >= k and inc_end[b + k - 1] >= k:
            return True
    return False

rng = random.Random(7)
for _ in range(2000):
    n = rng.randint(2, 20)
    k = rng.randint(1, n // 2)
    nums = [rng.randint(0, 4) for _ in range(n)]
    assert has_increasing_v2(nums, k) == oracle(nums, k)

# Bài 5
def count_valid_a(nums, k):
    n = len(nums)
    inc = [1] * n
    for i in range(n - 2, -1, -1):
        if nums[i] < nums[i + 1]:
            inc[i] = inc[i + 1] + 1
    return sum(1 for a in range(n - 2 * k + 1) if inc[a] >= k and inc[a + k] >= k)

assert count_valid_a([1, 2, 3, 4, 5, 6, 7], 3) == 2
assert count_valid_a([1, 2, 3, 1, 2, 3], 3) == 1
assert count_valid_a([5, 4, 3, 2], 2) == 0
print("Tất cả đáp án đều đúng.")

Tất cả đáp án đều đúng.


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (một lượt: độ dài đoạn tăng hiện tại, đoạn trước, và max)

```python
class Solution:
    def hasIncreasingSubarrays(self, nums: List[int], k: int) -> bool:
        mx = pre = cur = 0
        for i, x in enumerate(nums):
            cur += 1
            if i == len(nums) - 1 or x >= nums[i + 1]:
                mx = max(mx, cur // 2, min(pre, cur))
                pre, cur = cur, 0
        return mx >= k
```

**Checklist**

- [ ] Tăng **ngặt** (`<`), điểm ngắt là `>=`?
- [ ] Có cả hai trường hợp: cắt đôi một đoạn (`cur // 2`) **và** hai đoạn kề nhau (`min(pre, cur)`)?
- [ ] Đoạn **cuối** mảng được đóng (`i == len(nums) - 1`)?
- [ ] Sau khi đóng đoạn: `pre, cur = cur, 0`?
- [ ] Nếu dùng vét cạn: `range(n - 2 * k + 1)` (có `+ 1`)?
- [ ] Đã thử `k = 1`, `n = 2k`, mảng giảm dần, mảng có phần tử bằng nhau?
- [ ] Skeleton đúng `class Solution` / `def hasIncreasingSubarrays(self, nums: List[int], k: int) -> bool`?